In [34]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

ARTIFACT_DIR = PROJECT_ROOT / "data" / "processed" / "artifacts"

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Artifact directory:", ARTIFACT_DIR)
print("Directory exists:", ARTIFACT_DIR.exists())

Project root: d:\PG Study material\Foundations of AI\LSTM_explainer
Artifact directory: d:\PG Study material\Foundations of AI\LSTM_explainer\data\processed\artifacts
Directory exists: True


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

print("Libraries imported successfully!")

Libraries imported successfully!


In [5]:
PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "india_weather_merged.csv"

df = pd.read_csv(DATA_PATH)

df["date"] = pd.to_datetime(df["date"])

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (124700, 19)


,date,city,temperature_2m_max,temperature_2m_min,temperature_2m_mean,apparent_temperature_max,precipitation_sum,precipitation_hours,relative_humidity_2m_mean,wind_speed_10m_max,wind_direction_10m_dominant,shortwave_radiation_sum,et0_fao_evapotranspiration,weather_code,state,region,latitude,longitude,elevation_m
0,2020-01-01,Agartala,26.2,16.2,20.5,28.3,0.3,3.0,79,11.5,60,11.62,2.29,51,Tripura,Northeast,23.8315,91.2868,12.0
1,2020-01-02,Agartala,26.5,15.0,20.2,28.8,0.0,0.0,86,8.0,30,11.70,2.25,3,Tripura,Northeast,23.8315,91.2868,12.0
2,2020-01-03,Agartala,20.2,15.7,18.0,21.4,18.5,16.0,90,17.1,27,5.77,1.14,63,Tripura,Northeast,23.8315,91.2868,12.0
3,2020-01-04,Agartala,21.8,14.7,17.9,22.7,0.6,2.0,86,9.4,221,8.38,1.62,51,Tripura,Northeast,23.8315,91.2868,12.0
4,2020-01-05,Agartala,21.8,13.6,17.0,22.1,0.0,0.0,85,8.7,358,9.72,1.82,3,Tripura,Northeast,23.8315,91.2868,12.0


In [6]:
print("Number of cities:", df["city"].nunique())

print("Date range:")
print(df["date"].min(), "to", df["date"].max())

print("\nMissing values:")
print(df[["date", "city", "temperature_2m_max"]].isnull().sum())

print("\nRecords per city:")
print(df.groupby("city").size().describe())

Number of cities: 50
Date range:
2020-01-01 00:00:00 to 2026-10-29 00:00:00

Missing values:
date                  0
city                  0
temperature_2m_max    0
dtype: int64

Records per city:
count      50.0
mean     2494.0
std         0.0
min      2494.0
25%      2494.0
50%      2494.0
75%      2494.0
max      2494.0
dtype: float64


In [7]:
# Get all unique dates in chronological order
unique_dates = np.sort(df["date"].unique())

# Calculate split boundaries
total_dates = len(unique_dates)

train_end_idx = int(total_dates * 0.70)
val_end_idx = int(total_dates * 0.85)

train_end_date = unique_dates[train_end_idx - 1]
val_end_date = unique_dates[val_end_idx - 1]

# Split the dataset
train_df = df[df["date"] <= train_end_date].copy()

val_df = df[
    (df["date"] > train_end_date) &
    (df["date"] <= val_end_date)
].copy()

test_df = df[df["date"] > val_end_date].copy()

# Display results
print("Training period:")
print(train_df["date"].min(), "to", train_df["date"].max())

print("\nValidation period:")
print(val_df["date"].min(), "to", val_df["date"].max())

print("\nTesting period:")
print(test_df["date"].min(), "to", test_df["date"].max())

print("\nNumber of rows:")
print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

Training period:
2020-01-01 00:00:00 to 2024-10-10 00:00:00

Validation period:
2024-10-11 00:00:00 to 2025-10-19 00:00:00

Testing period:
2025-10-20 00:00:00 to 2026-10-29 00:00:00

Number of rows:
Training: 87250
Validation: 18700
Testing: 18750


In [9]:
from sklearn.preprocessing import MinMaxScaler

In [10]:
# Create the scaler
scaler = MinMaxScaler(feature_range=(0, 1))

# Fit ONLY on training temperatures
scaler.fit(train_df[["temperature_2m_max"]])

# Transform all three datasets using the same scaler
train_df["temperature_scaled"] = scaler.transform(
    train_df[["temperature_2m_max"]]
)

val_df["temperature_scaled"] = scaler.transform(
    val_df[["temperature_2m_max"]]
)

test_df["temperature_scaled"] = scaler.transform(
    test_df[["temperature_2m_max"]]
)

print("Original training temperature range:")
print(train_df["temperature_2m_max"].min(),
      train_df["temperature_2m_max"].max())

print("\nScaler parameters:")
print("Minimum:", scaler.data_min_[0])
print("Maximum:", scaler.data_max_[0])

print("\nScaled training temperature range:")
print(train_df["temperature_scaled"].min(),
      train_df["temperature_scaled"].max())

print("\nScaled validation temperature range:")
print(val_df["temperature_scaled"].min(),
      val_df["temperature_scaled"].max())

print("\nScaled testing temperature range:")
print(test_df["temperature_scaled"].min(),
      test_df["temperature_scaled"].max())

Original training temperature range:
-12.6 48.3

Scaler parameters:
Minimum: -12.6
Maximum: 48.3

Scaled training temperature range:
0.0 1.0

Scaled validation temperature range:
0.0443349753694581 0.9540229885057472

Scaled testing temperature range:
0.06732348111658457 0.9425287356321839


In [17]:
# Add normalized temperatures to the original dataframe
df["temperature_scaled"] = scaler.transform(
    df[["temperature_2m_max"]]
)

print(df[["city", "date", "temperature_2m_max", "temperature_scaled"]].head())

       city       date  temperature_2m_max  temperature_scaled
0  Agartala 2020-01-01                26.2            0.637110
1  Agartala 2020-01-02                26.5            0.642036
2  Agartala 2020-01-03                20.2            0.538588
3  Agartala 2020-01-04                21.8            0.564860
4  Agartala 2020-01-05                21.8            0.564860


In [18]:
SEQUENCE_LENGTH = 30

X_train, y_train = [], []
X_val, y_val = [], []
X_test, y_test = [], []

for city in df["city"].unique():

    # Get this city's complete chronological history
    city_data = df[df["city"] == city].sort_values("date")

    # Extract scaled temperatures
    values = city_data["temperature_scaled"].values

    # Get chronological boundaries
    train_end = len(train_df[train_df["city"] == city])
    val_end = train_end + len(val_df[val_df["city"] == city])

    # Generate sequences for the complete city history
    X_city, y_city = create_sequences(values, SEQUENCE_LENGTH)

    # Target date positions
    target_positions = np.arange(
        SEQUENCE_LENGTH, len(values)
    )

    # Assign samples based on target dates
    train_mask = target_positions < train_end

    val_mask = (
        (target_positions >= train_end) &
        (target_positions < val_end)
    )

    test_mask = target_positions >= val_end

    # Append samples to respective datasets
    X_train.extend(X_city[train_mask])
    y_train.extend(y_city[train_mask])

    X_val.extend(X_city[val_mask])
    y_val.extend(y_city[val_mask])

    X_test.extend(X_city[test_mask])
    y_test.extend(y_city[test_mask])

# Convert to NumPy arrays
X_train = np.array(X_train)
y_train = np.array(y_train)

X_val = np.array(X_val)
y_val = np.array(y_val)

X_test = np.array(X_test)
y_test = np.array(y_test)

In [19]:
print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)

Training: (85750, 30) (85750,)
Validation: (18700, 30) (18700,)
Testing: (18750, 30) (18750,)


In [20]:
# Add the feature dimension
X_train = X_train.reshape(-1, 30, 1)
X_val = X_val.reshape(-1, 30, 1)
X_test = X_test.reshape(-1, 30, 1)

# Reshape targets
y_train = y_train.reshape(-1, 1)
y_val = y_val.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)

Training: (85750, 30, 1) (85750, 1)
Validation: (18700, 30, 1) (18700, 1)
Testing: (18750, 30, 1) (18750, 1)


In [21]:
import torch
from torch.utils.data import TensorDataset, DataLoader

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.15.0+cu132
CUDA available: True


In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Selected device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Selected device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [23]:
# Create a unique ID for every city
city_to_id = {
    city: idx
    for idx, city in enumerate(df["city"].unique())
}

print("Number of cities:", len(city_to_id))
print(city_to_id)

Number of cities: 50
{'Agartala': 0, 'Agra': 1, 'Ahmedabad': 2, 'Amritsar': 3, 'Bengaluru': 4, 'Bhopal': 5, 'Bhubaneswar': 6, 'Chandigarh': 7, 'Chennai': 8, 'Coimbatore': 9, 'Dehradun': 10, 'Delhi': 11, 'Gangtok': 12, 'Guwahati': 13, 'Gwalior': 14, 'Hyderabad': 15, 'Imphal': 16, 'Indore': 17, 'Jaipur': 18, 'Jammu': 19, 'Jodhpur': 20, 'Kanpur': 21, 'Kochi': 22, 'Kolkata': 23, 'Leh': 24, 'Lucknow': 25, 'Ludhiana': 26, 'Madurai': 27, 'Mangaluru': 28, 'Mumbai': 29, 'Mysuru': 30, 'Nagpur': 31, 'Nashik': 32, 'Panaji': 33, 'Patna': 34, 'Port Blair': 35, 'Prayagraj': 36, 'Puducherry': 37, 'Pune': 38, 'Raipur': 39, 'Ranchi': 40, 'Shillong': 41, 'Shimla': 42, 'Srinagar': 43, 'Surat': 44, 'Thiruvananthapuram': 45, 'Vadodara': 46, 'Varanasi': 47, 'Vijayawada': 48, 'Visakhapatnam': 49}


In [24]:
SEQUENCE_LENGTH = 30

X_train, y_train, city_train = [], [], []
X_val, y_val, city_val = [], [], []
X_test, y_test, city_test = [], [], []

for city in df["city"].unique():

    city_data = df[df["city"] == city].sort_values("date")

    values = city_data["temperature_scaled"].values

    city_id = city_to_id[city]

    train_end = len(train_df[train_df["city"] == city])
    val_end = train_end + len(val_df[val_df["city"] == city])

    X_city, y_city = create_sequences(values, SEQUENCE_LENGTH)

    target_positions = np.arange(SEQUENCE_LENGTH, len(values))

    train_mask = target_positions < train_end

    val_mask = (
        (target_positions >= train_end) &
        (target_positions < val_end)
    )

    test_mask = target_positions >= val_end

    # Training
    X_train.extend(X_city[train_mask])
    y_train.extend(y_city[train_mask])
    city_train.extend(np.full(train_mask.sum(), city_id))

    # Validation
    X_val.extend(X_city[val_mask])
    y_val.extend(y_city[val_mask])
    city_val.extend(np.full(val_mask.sum(), city_id))

    # Testing
    X_test.extend(X_city[test_mask])
    y_test.extend(y_city[test_mask])
    city_test.extend(np.full(test_mask.sum(), city_id))


# Convert to NumPy arrays
X_train = np.array(X_train).reshape(-1, 30, 1)
y_train = np.array(y_train).reshape(-1, 1)
city_train = np.array(city_train)

X_val = np.array(X_val).reshape(-1, 30, 1)
y_val = np.array(y_val).reshape(-1, 1)
city_val = np.array(city_val)

X_test = np.array(X_test).reshape(-1, 30, 1)
y_test = np.array(y_test).reshape(-1, 1)
city_test = np.array(city_test)

In [25]:
print("Training:")
print("X:", X_train.shape)
print("y:", y_train.shape)
print("City IDs:", city_train.shape)

print("\nValidation:")
print("X:", X_val.shape)
print("y:", y_val.shape)
print("City IDs:", city_val.shape)

print("\nTesting:")
print("X:", X_test.shape)
print("y:", y_test.shape)
print("City IDs:", city_test.shape)

Training:
X: (85750, 30, 1)
y: (85750, 1)
City IDs: (85750,)

Validation:
X: (18700, 30, 1)
y: (18700, 1)
City IDs: (18700,)

Testing:
X: (18750, 30, 1)
y: (18750, 1)
City IDs: (18750,)


In [26]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Temperature sequences
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

# Target temperatures
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# City IDs must be integers
city_train_tensor = torch.tensor(city_train, dtype=torch.long)
city_val_tensor = torch.tensor(city_val, dtype=torch.long)
city_test_tensor = torch.tensor(city_test, dtype=torch.long)

In [27]:
print("Temperature tensor:", X_train_tensor.dtype)
print("Target tensor:", y_train_tensor.dtype)
print("City ID tensor:", city_train_tensor.dtype)

Temperature tensor: torch.float32
Target tensor: torch.float32
City ID tensor: torch.int64


In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Selected device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Selected device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [29]:
train_dataset = TensorDataset(
    X_train_tensor,
    city_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    city_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    city_test_tensor,
    y_test_tensor
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

Training samples: 85750
Validation samples: 18700
Testing samples: 18750


In [30]:
BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training batches: 670
Validation batches: 147
Testing batches: 147


In [31]:
sample_X, sample_city, sample_y = next(iter(train_loader))

print("Input batch shape:", sample_X.shape)
print("City ID batch shape:", sample_city.shape)
print("Target batch shape:", sample_y.shape)

Input batch shape: torch.Size([128, 30, 1])
City ID batch shape: torch.Size([128])
Target batch shape: torch.Size([128, 1])


In [35]:
import json
import joblib

# Save scaler
joblib.dump(
    scaler,
    ARTIFACT_DIR / "temperature_scaler.pkl"
)

# Save city mapping
with open(ARTIFACT_DIR / "city_to_id.json", "w") as file:
    json.dump(city_to_id, file, indent=4)

print("Scaler and city mapping saved!")

Scaler and city mapping saved!


In [36]:
np.savez_compressed(
    ARTIFACT_DIR / "weather_sequences.npz",

    X_train=X_train,
    y_train=y_train,
    city_train=city_train,

    X_val=X_val,
    y_val=y_val,
    city_val=city_val,

    X_test=X_test,
    y_test=y_test,
    city_test=city_test
)

print("All sequence datasets saved!")

All sequence datasets saved!


In [37]:
config = {
    "sequence_length": SEQUENCE_LENGTH,
    "number_of_cities": len(city_to_id),
    "input_features": 1,
    "target": "temperature_2m_max",
    "normalization": "MinMaxScaler",
    "batch_size": BATCH_SIZE
}

with open(ARTIFACT_DIR / "config.json", "w") as file:
    json.dump(config, file, indent=4)

print("Configuration saved!")

Configuration saved!


In [38]:
print("Saved artifacts:")

for file in ARTIFACT_DIR.iterdir():
    print(file.name, "-", round(file.stat().st_size / (1024 * 1024), 2), "MB")

Saved artifacts:
city_to_id.json - 0.0 MB
config.json - 0.0 MB
temperature_scaler.pkl - 0.0 MB
weather_sequences.npz - 0.84 MB
